Select all trades since 2024

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time
import datetime as dt

from Database.DB_reader import Database

from datetime import datetime, time
from Database.TPData import TPData, TPDataDa
from OrderBook.OrderBook import OrderBookSnaps
from SynthSpread.spreadviewer_class import SpreadSingle
from Strategies.Sparse_momentum.ob_attributes import OB_attributes
from Utilities.excel_loaders import conn_out_xload
from Utilities.Storage import get_curr_storage_path
from Database.Timescale import utils

In [ ]:
# 1. Read data from source DB
conn = Database('timescaledb')

query=f"""select distinct datetime, nanotime, tradeid from  public.trades 
          where datetime>='2025-04-01' and datetime<='2025-05-14' 
          and EXTRACT(HOUR FROM datetime) BETWEEN 8 AND 18
          and instid in ('10641710', '10001075', '10100480', '10012528', '10002806')
          order by datetime asc, nanotime asc""" # where rownum <= 100"""
timestamps=conn.execute(query)

print(f"✅ Loaded {len(timestamps)} rows from source database.")

Connected to the database timescaledb
Disconnected from the database timescaledb
✅ Loaded 544484 rows from source database.
Disconnected from the database timescaledb
✅ Loaded 544484 rows from source database.


In [ ]:
query=f"""select distinct datetime, nanotime, tradeid, price from  public.trades 
          where datetime>='2025-04-01' and datetime<='2025-05-14' 
          and EXTRACT(HOUR FROM datetime) BETWEEN 8 AND 18
          and instid in ('10641710')
          and firstsequenceid in ('10000106')
          and firstsequenceitemid in ('23')
          and secondsequenceitemid in ('0')
          and aggressorcompany_id_ut in ('14')
          order by datetime asc, nanotime asc""" # where rownum <= 100"""
df_trades=conn.execute(query)

print(f"✅ Loaded {len(df_trades)} rows from source database.")

Connected to the database timescaledb
Disconnected from the database timescaledb
✅ Loaded 15563 rows from source database.
Disconnected from the database timescaledb
✅ Loaded 15563 rows from source database.


In [ ]:
# df_trades2 = utils.get_trades_data('dey1', '2025-01-01', '2025-05-14')

In [ ]:
# test = timestamps.merge(df_trades2[['tradeid', 'price']], how='inner', on=['tradeid'], suffixes=('', '_y'))

In [ ]:
# Merge trades to timestamps
trades = timestamps.merge(df_trades, on=['datetime', 'nanotime', 'tradeid'], how='left').ffill()

In [ ]:
# Function to generate OHLC values for each timestamp using cumulative rolling within groups
def generate_ohlc_for_timestamps(df, granularity='1H'):
    # Drop rows with missing prices
    df = df.dropna(subset=['price']).copy()
    
    # Ensure 'price' is numeric
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    
    # Assign each trade to a period based on granularity
    df['period'] = df['datetime'].dt.floor(granularity)
    
    # Define a function to compute cumulative OHLC within a group
    def compute_cumulative_ohlc(group):
        group = group.copy()  # Avoid SettingWithCopyWarning
        group['open'] = group['price'].expanding().apply(lambda x: x.iloc[0], raw=False)
        group['high'] = group['price'].expanding().max()
        group['low'] = group['price'].expanding().min()
        group['close'] = group['price']
        group['candle_start'] = group['period']  # Add start datetime of the candle
        return group
    
    # Apply the cumulative OHLC computation within each group
    df = df.groupby('period', group_keys=False).apply(compute_cumulative_ohlc)
    
    return df

In [ ]:
# Cell 9

In [ ]:
# Function to generate candles
def generate_candles(df, granularity='5T'):
    # Ensure price is numeric and drop NaN prices
    df = df.copy()
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df = df.dropna(subset=['price'])
    ohlc_dict = {
        'price': ['first', 'max', 'min', 'last']
    }
    candles = df.resample(granularity).apply(ohlc_dict)
    candles.columns = ['open', 'high', 'low', 'close']
    return candles.dropna()

In [ ]:
# Set the granularity for the OHLC data
higher_timeframe = '1h'
lower_timeframe = '15min'
# Generate higher granularity OHLC data
candles_high = generate_candles(trades.set_index('datetime')[['price']], granularity=higher_timeframe)
candles_low = generate_candles(trades.set_index('datetime')[['price']], granularity=lower_timeframe)
# Generate OHLC data for each timestamp using cumulative rolling
df_higher = generate_ohlc_for_timestamps(trades, granularity=higher_timeframe)
df_lower = generate_ohlc_for_timestamps(trades, granularity=lower_timeframe)

C:\Users\krajcovic\AppData\Local\Temp\ipykernel_28164\757995010.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('period', group_keys=False).apply(compute_cumulative_ohlc)
C:\Users\krajcovic\AppData\Local\Temp\ipykernel_28164\757995010.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('period', group_keys=False).apply(compute_cumulative_ohlc)
C:\Users\krajcovic\AppData

In [ ]:
# Compute ATR for each row using lookback from candles (excluding current period)
def compute_atr_from_candles(df, candles, period=14):
    # Ensure index is datetime for candles
    candles = candles.copy()
    candles.index = pd.to_datetime(candles.index)
    # Prepare result column
    atrs = []
    for idx, row in df.iterrows():
        curr_time = row['period'] if 'period' in row else row['datetime']
        # Get lookback window (exclude current period)
        lookback = candles.loc[candles.index < curr_time].tail(period)
        if len(lookback) < period:
            atrs.append(np.nan)
            continue
        prev_close = lookback['close'].shift(1)
        tr = pd.DataFrame({
            'high': lookback['high'],
            'low': lookback['low'],
            'prev_close': prev_close
        })
        tr['tr'] = tr.apply(lambda r: max(
            r['high'] - r['low'],
            abs(r['high'] - r['prev_close']) if not pd.isna(r['prev_close']) else 0,
            abs(r['low'] - r['prev_close']) if not pd.isna(r['prev_close']) else 0
        ), axis=1)
        atrs.append(tr['tr'].mean())
    df = df.copy()
    df['atr'] = atrs
    return df

In [ ]:
# Example usage for higher timeframe:
df_higher_with_atr = compute_atr_from_candles(df_higher, candles_high, period=14)


In [ ]:
df_higher_with_atr.columns

Index(['datetime', 'nanotime', 'tradeid', 'price', 'period', 'open', 'high',
       'low', 'close', 'candle_start', 'atr'],
      dtype='object')

In [ ]:
df_higher_with_atr[['datetime', 'atr']].set_index('datetime').plot()

<Axes: xlabel='datetime'>

In [ ]:
def detect_swing_points(ohlc_df: pd.DataFrame,
                        high_col: str = 'high',
                        low_col: str = 'low') -> pd.DataFrame:
    """
    Compute swing highs/lows based only on the given DataFrame (no external candles).
    Returns DataFrame with 'swing_high' and 'swing_low' columns.
    """
    df = ohlc_df.copy().reset_index()
    df['min'] = np.nan
    df['max'] = np.nan
    for idx, row in df.iterrows():
        h = row[high_col]
        l = row[low_col]
        if idx == 0:
            last_min = [0, l]
            last_min2 = [0, l]
            last_max = [0, h]
            last_max2 = [0, h]
            df.at[idx, 'min'] = l
            df.at[idx, 'max'] = h
            continue
        # distances since last swings
        dM = idx - last_max[0]
        dm = idx - last_min[0]
        # new max
        if h > last_max[1]:
            df.at[idx, 'max'] = h
            if dM > 1:
                last_max2 = last_max
                last_max = [idx, h]
                slice_low = df.loc[last_max2[0]: last_max[0], low_col]
                new_min = slice_low.min()
                new_min_idx = slice_low.idxmin()
                last_min2 = last_min
                last_min = [new_min_idx, new_min]
                df.at[idx, 'min'] = new_min
            else:
                last_max = [idx, h]
        # new min
        if l < last_min[1]:
            df.at[idx, 'min'] = l
            if dm > 1:
                last_min2 = last_min
                last_min = [idx, l]
                slice_high = df.loc[last_min2[0]: last_min[0], high_col]
                new_max = slice_high.max()
                new_max_idx = slice_high.idxmax()
                last_max2 = last_max
                last_max = [new_max_idx, new_max]
                df.at[idx, 'max'] = new_max
            else:
                last_min = [idx, l]
    swings = pd.DataFrame({
        'swing_high': df['max'].values,
        'swing_low':  df['min'].values
    }, index=ohlc_df.index)
    return swings.ffill()

In [ ]:
# Cell 17
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import MultiCursor

def plot_price_with_swings(df, swing_high_col='swing_high', swing_low_col='swing_low', max_rows=500):
    """
    Plot only price candles and swing points (no other charts), in an interactive window.
    Plots up to `max_rows` most recent periods for performance.
    """
    df_last = df.groupby('period').apply(lambda g: g.tail(1)).dropna(subset=['open', 'high', 'low', 'close'])
    if len(df_last) > max_rows:
        df_last = df_last.iloc[-max_rows:].copy()
    n = len(df_last)
    x = np.arange(n)
    width = 0.6

    fig, ax = plt.subplots(figsize=(12, 6))

    # Candlesticks
    for i, (_, row) in enumerate(df_last.iterrows()):
        o, h, l, c = row['open'], row['high'], row['low'], row['close']
        ax.vlines(i, l, h, color='black', linewidth=1)
        ax.bar(i, c-o, width, bottom=o,
               color='green' if c > o else 'red', edgecolor='black')

    # Swing points
    ax.plot(x, df_last[swing_high_col], '.', ms=8, color='blue',  label=swing_high_col)
    ax.plot(x, df_last[swing_low_col], '.', ms=8, color='purple', label=swing_low_col)

    ax.set_title('Price Candles with Swing Levels')
    ax.set_ylabel('Price')
    ax.legend(loc='upper left')
    pad = (df_last['high'].max() - df_last['low'].min()) * 0.05
    ax.set_ylim(df_last['low'].min() - pad, df_last['high'].max() + pad)
    N = max(n // 10, 1)
    ticks  = x[::N]
    labels = df_last['datetime'].dt.strftime('%Y-%m-%d %H:%M')[::N]
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.grid(True)

    # Interactive crosshair
    multi = MultiCursor(
        fig.canvas,
        (ax,),
        horizOn=False,
        vertOn=True,
        color='gray',
        lw=1
    )

    def fmt_coord(xi, y):
        idx = int(round(xi))
        if 0 <= idx < len(df_last):
            return f"{df_last['datetime'].iloc[idx]:%Y-%m-%d %H:%M}, {y:.2f}"
        return f"{y:.2f}"
    ax.format_coord = fmt_coord

    plt.tight_layout()
    plt.show()

# Example usage:
# plot_price_with_swings(df_lower, max_rows=500)

In [ ]:
candles_high

,open,high,low,close
datetime,,,,
2025-04-01 08:00:00,85.60,85.75,85.50,85.50
2025-04-01 09:00:00,85.50,85.79,85.25,85.38
2025-04-01 10:00:00,85.38,85.50,85.20,85.22
2025-04-01 11:00:00,85.22,86.49,85.05,86.49
2025-04-01 12:00:00,86.49,86.89,86.25,86.50
...,...,...,...,...
2025-05-13 14:00:00,89.80,89.95,89.32,89.80
2025-05-13 15:00:00,89.80,90.20,89.80,90.15
2025-05-13 16:00:00,90.15,90.64,89.80,89.92


In [ ]:
swings = detect_swing_points(candles_low)
swings = swings.reset_index()
swings.rename(columns={'datetime': 'period'}, inplace=True)
df_lower_s = df_lower.merge(swings, on='period', how='left')
df_lower_s.tail()

,datetime,nanotime,tradeid,price,period,open,high,low,close,candle_start,swing_high,swing_low
544298,2025-05-13 18:56:00,730000000,17926358,89.95,2025-05-13 18:45:00,89.95,89.95,89.95,89.95,2025-05-13 18:45:00,91.05,89.32
544299,2025-05-13 18:56:06,621000000,17926360,89.95,2025-05-13 18:45:00,89.95,89.95,89.95,89.95,2025-05-13 18:45:00,91.05,89.32
544300,2025-05-13 18:56:06,621000000,17926362,89.95,2025-05-13 18:45:00,89.95,89.95,89.95,89.95,2025-05-13 18:45:00,91.05,89.32
544301,2025-05-13 18:57:15,330000000,17926365,89.95,2025-05-13 18:45:00,89.95,89.95,89.95,89.95,2025-05-13 18:45:00,91.05,89.32
544302,2025-05-13 18:57:15,534000000,17926368,89.95,2025-05-13 18:45:00,89.95,89.95,89.95,89.95,2025-05-13 18:45:00,91.05,89.32


In [ ]:
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import numpy as np

def plot_price_with_swings_popup(df, swing_high_col='swing_high', swing_low_col='swing_low', max_rows=500):
    """
    Plot price candles and swing points in a pop-up window with zoom/pan enabled.
    """
    df_last = df.groupby('period').apply(lambda g: g.tail(1)).dropna(subset=['open','high','low','close'])
    if len(df_last) > max_rows:
        df_last = df_last.iloc[-max_rows:].copy()
    n = len(df_last)
    x = np.arange(n)
    width = 0.6

    fig, ax = plt.subplots(figsize=(12, 6))

    for i, (_, row) in enumerate(df_last.iterrows()):
        o, h, l, c = row['open'], row['high'], row['low'], row['close']
        ax.vlines(i, l, h, color='black', linewidth=1)
        ax.bar(i, c-o, width, bottom=o,
               color='green' if c > o else 'red', edgecolor='black')

    ax.plot(x, df_last[swing_high_col], '.', ms=8, color='blue',  label=swing_high_col)
    ax.plot(x, df_last[swing_low_col], '.', ms=8, color='purple', label=swing_low_col)

    ax.set_title('Price Candles with Swing Levels (popup)')
    ax.set_ylabel('Price')
    ax.legend(loc='upper left')
    pad = (df_last['high'].max() - df_last['low'].min()) * 0.05
    ax.set_ylim(df_last['low'].min() - pad, df_last['high'].max() + pad)
    N = max(n // 10, 1)
    ticks = x[::N]
    labels = df_last['datetime'].dt.strftime('%Y-%m-%d %H:%M')[::N]
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_price_with_swings_popup(df_lower_s, max_rows=500)

C:\Users\krajcovic\AppData\Local\Temp\ipykernel_28164\2756876550.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_last = df.groupby('period').apply(lambda g: g.tail(1)).dropna(subset=['open','high','low','close'])


In [ ]:
swings

,swing_high,swing_low
datetime,,
2025-04-01 08:45:00,85.75,85.50
2025-04-01 09:00:00,85.75,85.25
2025-04-01 09:15:00,85.79,85.25
2025-04-01 09:30:00,85.79,85.25
2025-04-01 09:45:00,85.79,85.25
...,...,...
2025-05-13 17:45:00,91.05,89.32
2025-05-13 18:00:00,91.05,89.32
2025-05-13 18:15:00,91.05,89.32


In [ ]:
def detect_swing_points_trade_data(trades_df: pd.DataFrame,
                                   high_col: str = 'high',
                                   low_col: str = 'low') -> pd.DataFrame:
    """
    Aggregates live trade data into candles by 'period', applies swing detection,
    and maps swing highs/lows back to each trade.
    """
    # 1. build one-row-per-candle DataFrame
    candles = (trades_df
               .groupby('period')
               .apply(lambda g: g.tail(1))
               .dropna(subset=['open','high','low','close']))
    candles = candles[['open','high','low','close']]

    # 2. detect swings on the candle-level ohlc
    swings = detect_swing_points(candles, high_col=high_col, low_col=low_col)

    # 3. join back to the original trades, forward-fill per trade
    result = trades_df.copy()
    result = result.join(swings, on='period', how='left')
    return result.ffill()

### Extract monthly slices of trade data

In [ ]:
# assume df_trades is time-indexed
monthly_trades = {}
for period in df_trades.index.to_period('M').unique():
    monthly_trades[str(period)] = df_trades[df_trades.index.to_period('M') == period]
    print(f"{period}: {len(monthly_trades[str(period)])} rows")
# now monthly_trades holds each month’s DataFrame